In [2]:
# Althought these instalations are in the requirements.txt this ensure the it's also in the kernel
%pip install langchain langchain-community faiss-cpu sentence-transformers PyMuPDF unidecode

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.9/73.9 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 25.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 26.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 483.4/483.4 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 28.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━

In [9]:
#Imports & setup
import os
import pickle
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Define paths relative to the project root
# This makes the code portable
DATA_PATH = "../data/rgeu2.pdf"
ARTIFACTS_PATH = "../artifacts"
VECTOR_STORE_PATH = os.path.join(ARTIFACTS_PATH, "vector_store")

# Ensure the artifacts directory exists
os.makedirs(ARTIFACTS_PATH, exist_ok=True)
os.makedirs(VECTOR_STORE_PATH, exist_ok=True)

print("Paths and libraries are ready.")

Paths and libraries are ready.


In [10]:
#Load and Chunk the document - src/ingest/extractor.py - src/ingest/chunker.py
print(f"Loading document from: {DATA_PATH}")
loader = PyMuPDFLoader(DATA_PATH)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(documents)

print(f"Document loaded and split into {len(docs)} chunks.")
print("Sample chunk content:\n---")
print(docs[0].page_content)
print("---")

Loading document from: ../data/rgeu2.pdf
Document loaded and split into 109 chunks.
Sample chunk content:
---
1 
 
 
 
 
 
A leitura deste documento, que transcreve o conteúdo do Decreto-Lei n.º 38 382, de 7 de Agosto de 
1951 (alterado pelos Decretos-Lei n.º 38 888 de 29 de Agosto de 1952, n.º 44 258 de 31 de Março de 
1962, n.º 45 027 de 13 de Maio de 1963, n.º 650/75 de 18 de Novembro, n.º 43/82 de 8 de 
Fevereiro, n.º 463/85 de 4 de Novembro, n.º172–H/86 de 30 de Junho, n.º 64/90 de 21 de Fevereiro, 
n.º 61/93 de 3 de Março, n.º 555/99 de 16 de Dezembro com as alterações introduzidas pelo 
Decreto-Lei n.º 177/2001 de 4 de Junho), não substitui a consulta da sua publicação em Diário da 
República. 
 
 
 
Decreto-Lei n.º 38 382 de 7 de Agosto de 1951 
Aprova o Regulamento geral das edificações urbanas  
Alterado pelos: 
 
Decreto-Lei n.º 38 888 de 29 de Agosto de 1952 
 
Decreto-Lei n.º 44 258 de 31 de Março de 1962 
 
Decreto-Lei n.º 45 027 de 13 de Maio de 1963 
 
Decreto-Lei n

In [11]:
#Create embeddings and build the vector store - src/embeddings/embedder.py - src/embeddings/vector_store.py
print("Initializing embedding model...")
# Using a multilingual model that performs well with Portuguese
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedder = HuggingFaceEmbeddings(model_name=model_name)

print("Building FAISS vector store...")
db = FAISS.from_documents(docs, embedder)

# Save the vector store to the artifacts directory
db.save_local(VECTOR_STORE_PATH)

print(f"✅ Vector store created and saved successfully at: {VECTOR_STORE_PATH}")

Initializing embedding model...
Building FAISS vector store...
✅ Vector store created and saved successfully at: ../artifacts/vector_store
